In [57]:
import particles_mod.Icosphere as ico
import numpy as np
import matplotlib.pyplot as plt
import libMobility as lm



In [58]:
radius = 2.0
density = 1.0

icosphere = ico.IcoSphere(radius, density)


06/06/2025 16:58:29 - VLMP - INFO - [VLMP] Starting VLMP 
06/06/2025 16:58:29 - VLMP - DEBUG - [VLMP] Processing "system" component ({'type': 'simulationName', 'parameters': {'simulationName': 'Icosphere'}}), type "simulationName" (VLMP.py:194)
06/06/2025 16:58:29 - VLMP - WARNING - [VLMP] (system) Component name not specified, using "simulationName". 
06/06/2025 16:58:29 - VLMP - DEBUG - [VLMP] (system) Component name "simulationName". (VLMP.py:200)
06/06/2025 16:58:29 - VLMP - DEBUG - [VLMP] (system) Component parameters "{'simulationName': 'Icosphere'}" (VLMP.py:207)
06/06/2025 16:58:29 - VLMP - DEBUG - [VLMP] Processing "system" component ({'type': 'simulationName', 'parameters': {'simulationName': 'Icosphere'}}), type "simulationName" (VLMP.py:194)
06/06/2025 16:58:29 - VLMP - WARNING - [VLMP] (system) Component name not specified, using "simulationName". 
06/06/2025 16:58:29 - VLMP - DEBUG - [VLMP] (system) Component name "simulationName". (VLMP.py:200)
06/06/2025 16:58:29 - VLMP

In [59]:

print("Number of particles:", icosphere.nparticles)
print("Radius od the icosphere:", icosphere.radius)

Number of particles: 42
Radius od the icosphere: 2.0


In [60]:
modes, eigenvalues = icosphere.obtain_modes()


Preliminary simulation structure created
Read Hessian file /tmp/tmpu4kjeb1a/hessian.txt with shape (1764, 11)


[WARNING] UAMMD-structured Python wrapper is not compatible with UAMMD-structured self restarting mechanism
[MESSAGE] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
[MESSAGE] ╻ ╻┏━┓┏┳┓┏┳┓╺┳┓
[MESSAGE] ┃ ┃┣━┫┃┃┃┃┃┃ ┃┃ Version: 2.5
[MESSAGE] ┗━┛╹ ╹╹ ╹╹ ╹╺┻┛
[MESSAGE] Compiled at: May 23 2025 13:23:39
[MESSAGE] Compiled in double precision mode
[MESSAGE] Computation started at Fri Jun  6 16:58:29 2025

[MESSAGE] [System] CUDA initialized
[MESSAGE] [System] Using device: NVIDIA GeForce RTX 3080 with id: 0
[MESSAGE] [System] Compute capability of the device: 8.6
[MESSAGE] ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ ━ 
[MESSAGE] [ExtendedSystem] (system) Name: Icosphere
[MESSAGE] [ExtendedSystem] (system) Seed: 1749221909411401422
[MESSAGE] [GlobalDataBase] Fundamental not specified, using default fundamental, "Time"
[WARNING] [Time] No timeStep specified, using 0.0 as default.
[MESSAGE] [Basic] Loaded type A, mass: 1.000000, radius: 0.500000, charge: 0.000000
[MES

In [ ]:



precision = np.float32 if lm.SelfMobility.precision == "float" else np.float64

def create_solver(model = "SelfMobility"):
    if model == "SelfMobility":
        solver = lm.SelfMobility("open", "open", "open")
        solver.setParameters(5)
    elif model == "NBody":
        solver = lm.NBody("open", "open", "open")
        solver.setParameters()
    
    
    solver.initialize(
        temperature=0.0,
        viscosity=1/(6 * np.pi ),
        hydrodynamicRadius=1.0,
        needsTorque=False,
    )
    return solver

def create_forces_from_modes(modes, eigenvalues):
    forces = - modes * eigenvalues.reshape((1, icosphere.nparticles * 3))
    return forces

def construct_mobility_matrix(modes, eigenvalues, solver):
    mobility_matrix = np.zeros((icosphere.nparticles * 3, icosphere.nparticles * 3), dtype=precision)
    forces = create_forces_from_modes(modes, eigenvalues)
    for mode_index in range(modes.shape[1]):
        force = forces[:, mode_index].reshape((icosphere.nparticles, 3))
        velocity = solver.Mdot(forces=force)[0]
        mobility_matrix[:, mode_index] = velocity.reshape((icosphere.nparticles * 3,))
    mobility_matrix = modes.T @ mobility_matrix
    return mobility_matrix
        


positions = icosphere.positions.astype(precision)
assert positions.shape == (icosphere.nparticles, 3)

solver = create_solver(model="NBody")
solver.setPositions(positions)

forces = create_forces_from_modes(modes, eigenvalues)

aux = modes.T @ forces + np.diag(eigenvalues)
assert np.all(np.isclose(aux, 0.0, atol=1e-10)), "Auxiliary check failed, not close to zero."

mobility_matrix = construct_mobility_matrix(modes, eigenvalues, solver)
print("Mobility matrix shape:", mobility_matrix.shape)

plt.imshow(mobility_matrix, cmap='hot', interpolation='nearest')
plt.colorbar()

RuntimeError: [Mobility] Invalid batch parameters for NBody. If in doubt, use the defaults.